In [ ]:
cd /Users/cl1704/PycharmProjects/embedding_dimension/RNN

In [ ]:
import os
import pandas as pd
import seaborn as sns
from matplotlib import gridspec
import matplotlib.pyplot as plt
import matplotlib
from Experiment_10.net import Net
import torch
from Experiment_10.ParametricWMTask import generate_trials
from sklearn.decomposition import PCA
import numpy as np
plt.rcParams["axes.grid"] = False
matplotlib.rcParams['axes.linewidth'] = .5

In [ ]:

u, z, mask, conditions = generate_trials(n_trials=1,n_conditions =25,n_t =400)


import numpy as np
import statsmodels.api as sm
import pandas as pd
#
net = Net(n=100, input_size=u.shape[2],output_size=u.shape[2],
          dale=False,
          sigma_in=0.,
          sigma_rec = 0.,
          base_in=0.1,
          lambda_std=0.4,
          beta=
          20)

net.alpha = .05
# Fit 4
dim = 4
net.fit(u,
        z,
        mask,
        lr=.001,
        epochs=1000,
        conditions=conditions,
        verbose=True,
        dim=dim,
        weight_decay=0.001,
        pca_target_1 = .2,
        pca_target_2=0,
        gain_target=0,
        kmeans_target = 0,
        lambda_kmeans = 0,
        lambda_gain_mean=.1,
        lambda_gain_std=1,
        lambda_pca_1=1,
        lambda_pca_2=1,
                            )

In [ ]:
%%sql


In [ ]:
##########################################
########## Plotting manifolds ############
##########################################
u, z, mask, conditions = generate_trials(n_trials=1,n_t = 400,n_conditions =400)
#net.alpha = .05
x = net(u)
x = (x - torch.mean(x,dim=(0,1)))/ torch.std(x,dim=(0,1))
x = x.detach().numpy()
pca = PCA().fit(x.reshape(-1,x.shape[2]))


rows = []
x = torch.tensor(x).float()
#net.recurrent_layer.weight.data = net.recurrent_layer.weight.data + .005*(2*torch.randn_like(net.recurrent_layer.weight.data)-1)
h = net.recurrent_layer(x)+net.input_layer(u)

flow_field = -x + net.activation(h)
e = torch.exp(net.beta * h)
b =net.beta* e/(1+e)**2
curvature =  (flow_field @ net.recurrent_layer.weight.data.t())**2 * b

x.shape

In [ ]:

for trial in range(x.shape[0]):
    trajectory = x[trial,:,:]
    proj_trajectory = pca.transform(trajectory)
    for t in range(x.shape[1]):

        rows.append({'t': t,
             'theta': conditions[trial]['theta'],
             'x': proj_trajectory[t,0],
             'y': proj_trajectory[t,1],
             'z': proj_trajectory[t,2],
             'c': torch.linalg.norm(curvature[trial,t,0]).item(),

              'b':torch.linalg.norm(b[trial,t,:]).item()})

new_df = pd.DataFrame(rows)


In [ ]:
fig = plt.figure(figsize=(8,1.25))
from scipy.spatial import ConvexHull
gs = gridspec.GridSpec(1,5)

times  = [2 ,5 , 15, 25]
for i in range(4):
    ax=fig.add_subplot(gs[0,i])
# sns.scatterplot(data = new_df[new_df.t<int(200*.4)], x = 'x', y = 'y', hue = 'b',palette = 'turbo',s=1,legend=False,alpha = .15,ax=ax,hue_norm=(0,7))
    sns.scatterplot(data = new_df[new_df.t==times[i]], x = 'y', y = 'z', hue = 'theta',s=1,legend=False,alpha = 1,ax=ax,palette ='husl')

    sns.scatterplot(data = new_df[new_df.t<int(200*.4)], x = 'y', y = 'z', color='gray',s=1,legend=False,alpha = .1,ax=ax)
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.axes.get_xaxis().set_visible(False)
    ax.axes.get_yaxis().set_visible(False)

ax=fig.add_subplot(gs[0,4])
sns.scatterplot(data = new_df[new_df.t<int(200*.4)], x = 'x', y = 'y', color='gray',s=1,legend=False,alpha = .1,ax=ax)
sns.scatterplot(data = new_df[new_df.t==times[i]], x = 'x', y = 'y', hue = 'theta',s=2,legend=False,alpha = 1,ax=ax,palette = 'husl')
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.spines['left'].set_visible(False)

ax.axes.get_xaxis().set_visible(False)
ax.axes.get_yaxis().set_visible(False)
plt.savefig('Figures/Figure_boundary_8.jpg',
            format='jpeg',
            dpi=450,
            bbox_inches='tight', transparent=True)
# ax.spines['left'].set_color('black')
# ax.spines['bottom'].set_color('black')


In [ ]:
fig = plt.figure(figsize=(2,2))
from scipy.spatial import ConvexHull
gs = gridspec.GridSpec(1,1)

ax0=fig.add_subplot(gs[0,0])

sns.scatterplot(data = new_df[new_df.t<int(200*.4)], x = 'y', y = 'z', hue = 'b',palette = 'turbo',s=2,legend=False,alpha = .25,ax=ax0,lw=0)

for ax in [ax0]:
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.axes.get_xaxis().set_visible(False)
    ax.axes.get_yaxis().set_visible(False)
    plt.savefig('Figures/Figure_boundary_8.jpg',
            format='jpeg',
            dpi=450,
            bbox_inches='tight', transparent=True)
# ax.spines['left'].set_color('black')
# ax.spines['bottom'].set_color('black')


In [ ]:
net.sigma_rec

In [ ]:
import numpy as np

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px


fig = make_subplots(
     rows=1, cols=1,
     specs=[[{"type": "scene"}]])

sub_df = new_df
fig.add_trace(go.Scatter3d(x=sub_df.x.values,
                          y=sub_df.y.values,
                          z=sub_df.z.values,
                              mode='markers',

                        marker=dict(
                            color=sub_df.b.values,
                            colorscale='turbo',
                            opacity=1,
                            size = 2
                        )), row=1, col=1)
#px.colors.diverging.balance
#
fig.update_layout( scene = dict(xaxis = dict(showgrid = False,showticklabels = False),
                                   yaxis = dict(showgrid = False,showticklabels = False),
                                   zaxis = dict(showgrid = False,showticklabels = False),

             ))
fig.update_scenes(xaxis_visible=False, yaxis_visible=False,zaxis_visible=False )

In [ ]:
from sklearn.manifold import TSNE
x = net(u).detach()
#x = (x - torch.mean(x,dim=(0,1)))/ torch.std(x,dim=(0,1))
hpoints = net.recurrent_layer(x) + net.input_layer(u)
flow_field = -x + net.activation(hpoints)
curvature = (flow_field @ net.recurrent_layer.weight.data.t())**2 * (torch.clamp(torch.exp(net.beta*hpoints.detach()),max = 1e6)/(1+torch.clamp(torch.exp(net.beta*hpoints.detach()),max = 1e6))**2)


In [ ]:
plt.figure()
sns.histplot(x = hpoints[:,:,0].detach().flatten())

In [ ]:
####################################################################
#################### CLUSTERING PLOT ###############################
####################################################################
net.sigma_rec = 0
x = net(u).detach()
#x = (x-torch.mean(x,dim=(0,1)))/torch.std(x,dim=(0,1))
h = net.recurrent_layer(x) + net.input_layer(u)
h = h - torch.mean(h, dim=(0, 1), keepdim=True)

z = x - torch.mean(x, dim=[0, 1], keepdim=True)
# dist = torch.var(z @ self.recurrent_layer.weight.t()) / torch.var(h, dim=(0, 1))

beta = torch.sum(h * z, dim=(0, 1)) / torch.sum(h ** 2, dim=(0, 1))
#
# h = torch.abs(torch.diff(x, dim=1) / (x[:, :-1, :] + 20))
# beta =  torch.mean(h, dim=(0, 1))

#x = (x - torch.mean(x,dim=(0,1)))/ torch.std(x,dim=(0,1))
#x = x  + 4*(2*torch.rand_like(x)-1)
x = x.numpy()
#
# net.sigma_rec = 0
# u, z, mask, conditions = generate_trials(n_trials=5,n_t = 75,n_conditions =25)
# x = net(u).detach().numpy()
#
# rows = []
# for trial in range(x.shape[0]):
#     rows.append({"trial": trial,
#                  'theta': conditions[trial]['theta'].item(),
#                  'y': x[trial, :, :]})
# response_df = pd.DataFrame(rows)
# new_df = response_df.groupby('theta').y.apply(lambda x: np.stack(x)).reset_index()
# response_matrix = np.stack(new_df.y.values)
# response_matrix = (response_matrix - np.mean(response_matrix, axis=(0,1,2)))/ np.std(response_matrix, axis=(0,1,2))
# response_matrix = np.mean(response_matrix,axis = 0)
# print(response_matrix.shape)
#

x_proj  = PCA().fit_transform(x.reshape(-1,x.shape[2]).T)
fig = plt.figure(figsize=(1.5,3))
from scipy.spatial import ConvexHull
gs = gridspec.GridSpec(2,1,hspace = .3)
ax0=fig.add_subplot(gs[0,0])
ax1=fig.add_subplot(gs[1,0])
sns.scatterplot(x = x_proj[:,0],y = x_proj[:,1],ax=ax1,s=2,hue= torch.abs(beta.detach()),alpha = 1,legend = False,palette ='turbo')
sns.scatterplot(x = x_proj[:,0],y = x_proj[:,1],ax=ax0,s=2,color='gray',alpha = .5,legend = False)
#sns.kdeplot(x = x_proj[:,0],y = x_proj[:,1],ax=ax0,color='gray',alpha = .5,legend = False)
for ax in [ax0,ax1]:
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.axes.get_xaxis().set_visible(False)
    ax.axes.get_yaxis().set_visible(False)
plt.savefig('Figures/Figure_1_c.jpg',
            format='jpeg',

            dpi=450,
            bbox_inches='tight', transparent=True)

In [ ]:
# Dimensionality
x = net(u)
x = (x - torch.mean(x,dim=(0,1)))/ torch.std(x,dim=(0,1))
x = x.detach().numpy()

pca = PCA().fit(x.reshape(-1,x.shape[2]))



fig = plt.figure(figsize=(1.25, 1.25))
from scipy.spatial import ConvexHull

gs = gridspec.GridSpec(1, 1)

ax = fig.add_subplot(gs[0, 0])
sns.barplot(x = np.arange(1,10),y = pca.explained_variance_ratio_[:9],color='red')
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)

ax.xaxis.set_tick_params(labelsize=6, bottom=True)
ax.yaxis.set_tick_params(labelsize=6, left=True)
ax.axes.get_xaxis().set_visible(False)

ax.set_ylim(0,.5)
plt.savefig('Figures/Figure_1_d.jpg',
            format='jpeg',

            dpi=450,
            bbox_inches='tight', transparent=True)


In [ ]:
torch.mean(beta)

In [ ]:


fig = plt.figure(figsize=(2, 2))
from scipy.spatial import ConvexHull

gs = gridspec.GridSpec(1, 1)
ax = fig.add_subplot(gs[0, 0])
sns.histplot(x = beta.detach(),ax=ax,color='black',stat = 'probability')
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.spines['left'].set_visible(False)
ax.set_xlabel('')
ax.set_xlim(0,1)

ax.xaxis.set_tick_params(labelsize=7, bottom=True)
ax.yaxis.set_tick_params(labelsize=6, left=True)
ax.axes.get_yaxis().set_visible(False)
plt.savefig('Figures/Figure_1_d.jpg',
            format='jpeg',
            dpi=450,
            bbox_inches='tight', transparent=True)


In [ ]:
###########################################################
################### Single neuron response ################
###########################################################
u, z, mask, conditions = generate_trials(n_trials=25,n_t = 75,n_conditions =20)
net.sigma_rec =0
x  = net(u)
x = (x  - torch.mean(x,dim=(0,1)))/ torch.std(x,dim=(0,1))
x = x.detach().numpy()


fig = plt.figure(figsize=(2,2))
from scipy.spatial import ConvexHull
gs = gridspec.GridSpec(1,1)

ax=fig.add_subplot(gs[0,0])


neuron = np.argsort(beta.detach())[2]
rows = []
for trial in range(x.shape[0]):
    for t in range(x.shape[1]):
        rows.append({'trial':trial,
                     't': t,
                         'theta': conditions[trial]['theta'].item(),
                         'x': x[trial,t, neuron]})
df = pd.DataFrame(rows)
# response_matrix = np.stack(df.sort_values('theta').x.values)
# sns.heatmap(response_matrix,ax = ax0,cmap='Grays',cbar = False)
sns.lineplot(data = df, x = 't', y = 'x',hue = 'theta',palette = 'husl',legend = False,lw = 1,errorbar='sd',ax=ax)
ax.xaxis.set_tick_params(labelsize=6, bottom=True)
ax.yaxis.set_tick_params(labelsize=6, left=True)

plt.savefig('Figures/Figure_1_d.jpg',
            format='jpeg',

            dpi=450,
            bbox_inches='tight', transparent=True)

In [ ]:
u, z, mask, conditions = generate_trials(n_trials=25,n_t = 75,n_conditions =20)
net.sigma_rec =0
x  = net(u).detach().numpy()
plt.figure()
sns.histplot(x = np.mean(x,axis=(0,1)))

In [ ]:
plt.figure()
sns.scatterplot(x = beta.detach(),y =np.mean(x,axis=(0,1)) )

In [ ]:
plt.figure()
sns.histplot(x = beta.detach())


In [ ]:
# Dimensionality
x = net(u)
#x = (x - torch.mean(x,dim=(0,1)))/ torch.std(x,dim=(0,1))
x = x.detach().numpy()

pca = PCA().fit(x.reshape(-1,x.shape[2]))



fig = plt.figure(figsize=(1.25, 1.25))
from scipy.spatial import ConvexHull

gs = gridspec.GridSpec(1, 1)

ax = fig.add_subplot(gs[0, 0])
sns.barplot(x = np.arange(1,11),y = pca.explained_variance_ratio_[:10],color='black')
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)

ax.xaxis.set_tick_params(labelsize=6, bottom=True)
ax.yaxis.set_tick_params(labelsize=6, left=True)
ax.axes.get_xaxis().set_visible(False)
plt.savefig('Figures/Figure_1_d.jpg',
            format='jpeg',

            dpi=450,
            bbox_inches='tight', transparent=True)


In [ ]:
x = net(u).detach()
sns.histplot(x = torch.std(x,dim=(0,1)))

In [ ]:
np.sum(pca.explained_variance_ratio_[dim:])

In [ ]:
d = torch.abs(hpoints) / torch.linalg.norm(net.recurrent_layer.weight.data,dim=1)

In [ ]:
plt.figure()
sns.scatterplot( y = curvature[:,:,0].detach().flatten(),x = d[:,:,0].detach().flatten())
sns.scatterplot( y = curvature[:,:,1].detach().flatten(),x = d[:,:,1].detach().flatten())
sns.scatterplot( y = curvature[:,:,2].detach().flatten(),x = d[:,:,2].detach().flatten())

In [ ]:

#net = torch.load('FigureModels/BoundariesandCurvature/model_2.pth',weights_only=False)
u, z, mask, conditions = generate_trials(n_trials=1,n_t = 75,n_conditions =200)
net.alpha = .2
x = net(u)
x = (x - torch.mean(x,dim=(0,1)))/ torch.std(x,dim=(0,1))
x = x.detach()

points = torch.tensor(x[:,:,:])
hpoints = net.recurrent_layer(points)+net.input_layer(u[:,:,:])
flow_field = -points + net.activation(hpoints)
curvature = (flow_field @ net.recurrent_layer.weight.data.t())**2 * (torch.clamp(torch.exp(net.beta*hpoints),max = 1e6)/(1+torch.clamp(torch.exp(net.beta*hpoints),max = 1e6))**2)

In [ ]:
torch.mean(curvature)

In [ ]:
c = (torch.clamp(torch.exp(net.beta*hpoints),max = 1e6)/(1+torch.clamp(torch.exp(net.beta*hpoints),max = 1e6))**2)
d = hpoints / torch.linalg.norm(net.recurrent_layer.weight.data,dim=1)

In [ ]:
x = net(u)
h = net.recurrent_layer(x) + net.input_layer(u)
h = h - torch.mean(h, dim=(0, 1), keepdim=True)
z = x - torch.mean(x, dim=[0, 1], keepdim=True)
# dist = torch.var(z @ self.recurrent_layer.weight.t()) / torch.var(h, dim=(0, 1))
d =  h / torch.linalg.norm(net.recurrent_layer.weight,dim=1)
d = d - torch.mean(d, dim=(0, 1), keepdim=True)
beta = torch.sum(h * z, dim=(0, 1)) / torch.sum(h ** 2, dim=(0, 1))

In [ ]:
plt.figure()
sns.histplot(x = beta.detach())

In [ ]:
fig = plt.figure(figsize=(1.5,1.5))
from scipy.spatial import ConvexHull
gs = gridspec.GridSpec(1,1)
ax=fig.add_subplot(gs[0,0])
for i in range(100):
    sns.scatterplot(x = d[:,:,i].flatten().detach(), y = h[:,:,i].flatten().detach(),s=2,color='tab:blue',ax=ax)

lims = [
            np.min([ax.get_xlim(), ax.get_ylim()]),  # min of both axes
            np.max([ax.get_xlim(), ax.get_ylim()]),  # max of both axes
        ]

# now plot both limits against eachother
ax.plot(lims, lims, 'k-', alpha=0.15, zorder=0,lw=.5)
ax.set_aspect('equal')
ax.set_xlabel("d",fontsize = 8)
ax.set_ylabel("h",fontsize = 8)

In [ ]:
x = net(u)
h = net.recurrent_layer(x) + net.input_layer(u)
c = torch.exp(net.beta*h)/(1+torch.exp(net.beta*h))**2

In [ ]:
fig = plt.figure(figsize=(1.5,1.5))
from scipy.spatial import ConvexHull
gs = gridspec.GridSpec(1,1)
ax=fig.add_subplot(gs[0,0])
neuron = 31
sns.scatterplot(x = h[:,:,neuron].flatten().detach(),y = c[:,:,neuron].flatten().detach(),ax=ax)
ax.set_ylim(0,.25)


In [ ]:
plt.figure()
sns.heatmap(net.recurrent_layer.weight.detach(),center=0)

In [ ]:
torch.linalg.norm(net.recurrent_layer.weight.data,dim=1)

In [ ]:
plt.figure()
sns.histplot(x = torch.mean(curvature,dim = (0,1)).detach().flatten(),stat = 'probability')

In [ ]:
plt.figure()
plt.plot(curvature[0,:,4].detach())
plt.plot(curvature[1,:,4].detach())
plt.plot(curvature[2,:,4].detach())
plt.plot(curvature[3,:,4].detach())

In [ ]:
h = net.recurrent_layer(x) + net.input_layer(u)
# z = x - torch.mean(x, dim=[0, 1], keepdim=True)
# dist = torch.var(z @ self.recurrent_layer.weight.t()) / torch.var(h, dim=(0, 1))
#    d =  h / torch.linalg.norm(self.recurrent_layer.weight,dim=1)
# dist = torch.sum(h * d, dim=(0, 1)) / torch.sum(d ** 2, dim=(0, 1))
dist = torch.var(x,dim=(0,1))/ torch.var(h, dim=(0, 1))

In [ ]:
plt.figure()
sns.histplot(x = dist.detach())

In [ ]:
torch.mean(dist)

In [ ]:
torch.std(torch.mean(curvature,dim = (0,1)))

In [ ]:
prediction = x.unsqueeze(3) + net.alpha*J(x) @ x.unsqueeze(3)

In [ ]:
x.shape,prediction.shape

In [ ]:
plt.figure()
sns.heatmap(J(x)[0,10].detach())

In [ ]:
J(x[:,:,:]).shape

In [ ]:
plt.figure()
sns.heatmap(J(x[0,10,:]).detach(),center=0)

In [ ]:
J(x[:,0,:])

In [ ]:
plt.figure()
for i in range(200):
    plt.plot(curvature[i,:,5].detach())

In [ ]:
x.max()

In [ ]:
print(net.recurrent_layer.bias)

In [ ]:
print(net.input_layer.bias)

In [ ]:
sns.color_palette("husl", 8)